# 🚞 Zero-shot RE Training

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
!git clone https://github.com/jackboyla/GLiNER.git
!cd GLiNER && pip install -e .
!python -m spacy download en_core_web_sm

In [ ]:
import os 

In [ ]:
os.chdir('./GLiNER')

In [ ]:
os.environ['TASK'] = 'rel'

In [ ]:
!python train.py --config config_small_rel.yaml --relation_extraction

In [6]:
from gliner import GLiNER

save_path = 'logs/model_100'
model = GLiNER.from_pretrained(save_path)
model

python(49413) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(49414) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(49415) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
config.json not found in /Users/jackboylan/Desktop/repos/GLiNER/logs/model_100
/Users/jackboylan/miniconda/envs/glirel/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:550: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


GLiNER(
  (token_rep_layer): TokenRepLayer(
    (bert_layer): TransformerWordEmbeddings(
      (model): DebertaV2Model(
        (embeddings): DebertaV2Embeddings(
          (word_embeddings): Embedding(128004, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
          (dropout): StableDropout()
        )
        (encoder): DebertaV2Encoder(
          (layer): ModuleList(
            (0-5): 6 x DebertaV2Layer(
              (attention): DebertaV2Attention(
                (self): DisentangledSelfAttention(
                  (query_proj): Linear(in_features=768, out_features=768, bias=True)
                  (key_proj): Linear(in_features=768, out_features=768, bias=True)
                  (value_proj): Linear(in_features=768, out_features=768, bias=True)
                  (pos_dropout): StableDropout()
                  (dropout): StableDropout()
                )
                (output): DebertaV2SelfOutput(
                  (dense): Linear(in_feature

In [7]:
import json
with open('./data/docred_expanded.jsonl', 'r') as f:
    data = [json.loads(line) for line in f]

i = 1

tokens = data[i]['tokenized_text']
ner = data[i]['ner']
labels = list(set([r['relation_text'] for r in data[i]['relations']]))
print(tokens)
print(ner)
print(labels)

['Skai', 'TV', 'is', 'a', 'Greek', 'free', '-', 'to', '-', 'air', 'television', 'network', 'based', 'in', 'Piraeus', '.', 'It', 'is', 'part', 'of', 'the', 'Skai', 'Group', ',', 'one', 'of', 'the', 'largest', 'media', 'groups', 'in', 'the', 'country', '.', 'It', 'was', 'relaunched', 'in', 'its', 'present', 'form', 'on', '1st', 'of', 'April', '2006', 'in', 'the', 'Athens', 'metropolitan', 'area', ',', 'and', 'gradually', 'spread', 'its', 'coverage', 'nationwide', '.', 'Besides', 'digital', 'terrestrial', 'transmission', ',', 'it', 'is', 'available', 'on', 'the', 'subscription', '-', 'based', 'encrypted', 'services', 'of', 'Nova', 'and', 'Cosmote', 'TV', '.', 'Skai', 'TV', 'is', 'also', 'a', 'member', 'of', 'Digea', ',', 'a', 'consortium', 'of', 'private', 'television', 'networks', 'introducing', 'digital', 'terrestrial', 'transmission', 'in', 'Greece', '.', 'At', 'launch', ',', 'Skai', 'TV', 'opted', 'for', 'dubbing', 'all', 'foreign', 'language', 'content', 'into', 'Greek', ',', 'instea

In [8]:
print(len(ner))
print(len(labels))

14
3


In [9]:
relations = model.predict_entities(tokens, labels, threshold=0.01, ner=ner)
print(relations)
print(len(relations))

[{'head_pos': [0, 2], 'tail_pos': [4, 5], 'head_text': ['Skai', 'TV'], 'tail_text': ['Greek'], 'label': 'headquarters location', 'score': 0.4080393314361572}, {'head_pos': [0, 2], 'tail_pos': [4, 5], 'head_text': ['Skai', 'TV'], 'tail_text': ['Greek'], 'label': 'owned by', 'score': 0.0800323635339737}, {'head_pos': [0, 2], 'tail_pos': [4, 5], 'head_text': ['Skai', 'TV'], 'tail_text': ['Greek'], 'label': 'country', 'score': 0.0742129236459732}, {'head_pos': [0, 2], 'tail_pos': [14, 15], 'head_text': ['Skai', 'TV'], 'tail_text': ['Piraeus'], 'label': 'headquarters location', 'score': 0.6513941287994385}, {'head_pos': [0, 2], 'tail_pos': [14, 15], 'head_text': ['Skai', 'TV'], 'tail_text': ['Piraeus'], 'label': 'owned by', 'score': 0.34687551856040955}, {'head_pos': [0, 2], 'tail_pos': [14, 15], 'head_text': ['Skai', 'TV'], 'tail_text': ['Piraeus'], 'label': 'country', 'score': 0.34535959362983704}, {'head_pos': [0, 2], 'tail_pos': [21, 23], 'head_text': ['Skai', 'TV'], 'tail_text': ['Skai

In [10]:
sorted_data_desc = sorted(relations, key=lambda x: x['score'], reverse=True)
print("\nDescending Order by Score:")
for item in sorted_data_desc:
    print(item)


Descending Order by Score:
{'head_pos': [0, 2], 'tail_pos': [14, 15], 'head_text': ['Skai', 'TV'], 'tail_text': ['Piraeus'], 'label': 'headquarters location', 'score': 0.6513941287994385}
{'head_pos': [4, 5], 'tail_pos': [14, 15], 'head_text': ['Greek'], 'tail_text': ['Piraeus'], 'label': 'headquarters location', 'score': 0.6362758874893188}
{'head_pos': [0, 2], 'tail_pos': [115, 116], 'head_text': ['Skai', 'TV'], 'tail_text': ['Greek'], 'label': 'headquarters location', 'score': 0.6359421014785767}
{'head_pos': [0, 2], 'tail_pos': [21, 23], 'head_text': ['Skai', 'TV'], 'tail_text': ['Skai', 'Group'], 'label': 'headquarters location', 'score': 0.6331073641777039}
{'head_pos': [21, 23], 'tail_pos': [115, 116], 'head_text': ['Skai', 'Group'], 'tail_text': ['Greek'], 'label': 'headquarters location', 'score': 0.624500036239624}
{'head_pos': [4, 5], 'tail_pos': [115, 116], 'head_text': ['Greek'], 'tail_text': ['Greek'], 'label': 'headquarters location', 'score': 0.6186618208885193}
{'head

In [ ]:
# How many did the model get right?

gt = [[r['head']['mention'], r['head']['position'], r['tail']['mention'], r['tail']['position'], r['relation_text']] for r in data[0]['relations']]
pred = [[r['head_text'], r['head_pos'], r['tail_text'], r['tail_pos'], r['label']] for r in relations]


# Function to compare entries
def compare_entries(entry_a, entry_b):
    # Compare head position, tail position, and label
    return entry_a[1] == entry_b[1] and entry_a[3] == entry_b[3] and entry_a[-1] == entry_b[-1]

# Find matching entries
matches = []
for entry_second in gt:
    for entry_first in pred:
        if compare_entries(entry_first, entry_second):
            matches.append(entry_second)

# Print matching entries from the second list that are found in the first list
print(len(matches), 'out of', len(relations), 'predictions and', len(gt), 'ground truths')
print(matches)